<a href="https://colab.research.google.com/github/rajajisai/leet-torch-submissions/blob/main/v3/classical-ml/kmeans/kmeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implement K-Means Clustering in PyTorch

**Difficulty**: 🟢 Easy

---

### Problem Statement

K-Means is one of the most widely used unsupervised learning algorithms. It partitions data into **K clusters** by iteratively assigning points to their nearest centroid and updating centroids to be the mean of their assigned points.

Your task is to implement K-Means clustering **from scratch** using only PyTorch tensor operations.

---

### Requirements

1. **Random Initialization** — Select K random data points as initial centroids.
2. **Assignment Step** — Assign each data point to the nearest centroid using Euclidean distance.
3. **Update Step** — Recompute each centroid as the mean of all points assigned to it.
4. **Convergence** — Stop when centroids no longer change (or change less than a tolerance).

---

### Constraints

- ✅ Use only PyTorch tensor operations (no sklearn, no Python loops over data points).
- ✅ Use `torch.cdist` or manual broadcasting for distance computation.
- ✅ Must work for any K and any dimensionality of data.
- ❌ Do **not** use any external clustering libraries.

---

<details>
  <summary>💡 Hint</summary>

  - Use `torch.cdist(data, centroids)` to compute pairwise distances efficiently. It returns a `(N, K)` distance matrix.
  - Use `torch.argmin(distances, dim=1)` to assign each point to its nearest centroid.
  - To update centroids, create a boolean mask for each cluster and compute `data[mask].mean(dim=0)`.
  - Check convergence by comparing old and new centroids with `torch.allclose()`.

</details>

---

In [1]:
import torch

In [2]:
# Generate synthetic data: 3 clusters in 2D
torch.manual_seed(42)

n_points_per_cluster = 100
true_centers = torch.tensor([[0.0, 0.0], [5.0, 5.0], [-5.0, 5.0]])

cluster_0 = torch.randn(n_points_per_cluster, 2) * 0.5 + true_centers[0]
cluster_1 = torch.randn(n_points_per_cluster, 2) * 0.5 + true_centers[1]
cluster_2 = torch.randn(n_points_per_cluster, 2) * 0.5 + true_centers[2]

data = torch.cat([cluster_0, cluster_1, cluster_2], dim=0)  # (300, 2)
print("Data shape:", data.shape)
print("True centers:\n", true_centers)

Data shape: torch.Size([300, 2])
True centers:
 tensor([[ 0.,  0.],
        [ 5.,  5.],
        [-5.,  5.]])


In [3]:
def kmeans_pp_init(data, k):
    N = data.shape[0]
    centroids = data[torch.randint(0, N, (1,))]
    for _ in range(k - 1):
        d2 = torch.cdist(data, centroids).min(dim=1).values.pow(2)
        probs = d2 / d2.sum()
        idx = torch.multinomial(probs, 1)
        centroids = torch.cat([centroids, data[idx]], dim=0)
    return centroids.clone()

In [4]:
def kmeans(data: torch.Tensor, k: int, max_iters: int = 100, tol: float = 1e-4):
    """
    K-Means clustering using PyTorch.

    Args:
        data (Tensor): Input data of shape (N, D).
        k (int): Number of clusters.
        max_iters (int): Maximum number of iterations.
        tol (float): Convergence tolerance for centroid movement.

    Returns:
        centroids (Tensor): Final centroids of shape (K, D).
        assignments (Tensor): Cluster assignment for each point, shape (N,).
    """
    N, D = data.shape

    # TODO: Step 1 - Initialize centroids by randomly selecting K data points
    # Hint: use torch.randperm(N)[:k] to pick random indices
    centroids=kmeans_pp_init(data,k)

    for i in range(max_iters):
        centroids_old=centroids.clone()

        distances = torch.cdist(data,centroids)

        group= torch.argmin(distances,dim=1)

        for j in range(k):
            mask = (group == j)
            if mask.any():
              centroids[j] = data[mask].mean(dim=0)
            else:
              random_idx = torch.randint(0, N, (1,)).item
              centroids[j] = data[random_idx].clone()


        if torch.allclose(centroids, centroids_old,rtol=0.0, atol=tol):
            break

    return centroids, group

In [5]:
# Validation
centroids, assignments = kmeans(data, k=3)
print("Learned centroids:\n", centroids)
print("True centers:\n", true_centers)

# Check that each learned centroid is close to one of the true centers
# (order may differ, so we match each learned centroid to its nearest true center)
dists = torch.cdist(centroids, true_centers)  # (3, 3)
min_dists = dists.min(dim=1).values
print("\nMin distance from each learned centroid to nearest true center:", min_dists)

assert (min_dists < 1.0).all(), f"Centroids too far from true centers: {min_dists}"
print("\nConvergence test PASSED!")

# Check that each cluster has roughly the right number of points
for i in range(3):
    count = (assignments == i).sum().item()
    print(f"Cluster {i}: {count} points")
    assert count > 50, f"Cluster {i} has too few points: {count}"

print("\nAll tests passed!")

Learned centroids:
 tensor([[-0.0766,  0.1225],
        [-5.0110,  4.9879],
        [ 5.0865,  5.0450]])
True centers:
 tensor([[ 0.,  0.],
        [ 5.,  5.],
        [-5.,  5.]])

Min distance from each learned centroid to nearest true center: tensor([0.1445, 0.0163, 0.0974])

Convergence test PASSED!
Cluster 0: 100 points
Cluster 1: 100 points
Cluster 2: 100 points

All tests passed!
